# Corpus cleaning and NLP

Preserve punctuation and inflection for natural text generation. Lemmatization is a separate optional representation.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import markovify
from markovify.nlp import Corpus
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
OUT = ROOT / "notebooks" / "outputs"
OUT.mkdir(parents=True, exist_ok=True)
SEED = 42
print(markovify.__file__)

In [ ]:
from markovify.nlp import SpacyNLP, clean_text
raw = Corpus.load(OUT / "raw.jsonl")
nlp = SpacyNLP(language="es")
clean = raw.map(clean_text, html=False, urls=True, emails=True)
segmented = clean.split(nlp.sentences)
filtered = segmented.filter_length(min_words=5, max_words=100)
processed = filtered.deduplicate(case_sensitive=False)
if not len(processed):
    raise ValueError("Empty processed corpus")
counts = pd.Series({"raw":len(raw), "clean":len(clean), "segmented":len(segmented), "filtered":len(filtered), "unique":len(processed)})
display(counts)
counts.plot.bar(title="Entries at each processing stage")
plt.ylabel("Entries")
plt.show()
processed.save(OUT / "processed.jsonl")
processed.save(OUT / "processed.txt")
display(pd.DataFrame({"sentence":processed.texts[:10]}))

In [ ]:
# Optional lexical representation; not used for training.
RUN_LEMMATIZATION = False
if RUN_LEMMATIZATION:
    annotator = SpacyNLP(model="es_core_news_sm")
    lexical = processed.map(annotator.transform, lemmatize=True, remove_stopwords=True)
    lexical.save(OUT / "lexical.jsonl")